<a href="https://colab.research.google.com/github/ARanjan45/AI-Counsellor/blob/main/TwitterSentimentAnalysis_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Twitter Sentiment Analysis(NLP Project)

#Setting Up the Environment, Fetching and analysing the stopwords

In [ ]:
#installing kaggle
! pip install kaggle

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
# API TO FETCH THE DATASET FROM KAGGLE
!kaggle datasets download -d kazanova/sentiment140

Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other
100% 80.9M/80.9M [00:00<00:00, 110MB/s]



In [ ]:
#Extracting the compressed dataset
from zipfile import ZipFile
dataset = '/content/sentiment140.zip'

with ZipFile(dataset, 'r') as zip:
  zip.extractall()
  print('Dataset Extracted successfully')

Dataset Extracted successfully


In [ ]:
#IMPORTING THE DEPENDENCIES
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
#Printing the stopwords in english
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

#**DATA PROCESSING**

In [ ]:
#Loading the data from CSV File to a Pandas dataframe

twitter_data = pd.read_csv('/content/training.1600000.processed.noemoticon.csv', encoding = 'ISO-8859-1')

In [ ]:
# checking the number of Rows and Columns

twitter_data.shape

(1599999, 6)

In [ ]:
#printing first 5 rows of the dataframe
twitter_data.head()

,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [ ]:
#naming the columns and reading the dataset again

column_names = ['target', 'id', 'date', 'flag', 'user', 'text']
twitter_data = pd.read_csv('/content/training.1600000.processed.noemoticon.csv', names=column_names, encoding = 'ISO-8859-1')

In [ ]:
# checking the number of Rows and Columns after naming

twitter_data.shape

(1600000, 6)

In [ ]:
#printing first 5 rows of the dataframe after naming to check the correctness
twitter_data.head()

,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [ ]:
#Check for Missing Values
twitter_data.isnull().sum()

,0
target,0
id,0
date,0
flag,0
user,0
text,0


In [ ]:
#Checking for the distribution of target columns(To check for how many positive and how many negative tweets are there in the 1.6 Million Dataset)
twitter_data['target'].value_counts()

,count
target,
0,800000
4,800000


In [ ]:
#Converting the labels in target(0 -> Negative, 1 -> Positive)
twitter_data.replace({'target':{4:1}}, inplace=True)
print("Conversion Successful")

Conversion Successful


#What the data shows?


*   0 -> **Negative Tweets**
*   1 -> **Positive Tweets**



In [ ]:
#Checking the distribution across each targets after labelling
twitter_data['target'].value_counts()

,count
target,
0,800000
1,800000


#**STEMMING THE DATA** *(Using Porter Stemmer)*

**Stemming** is the process of reducing the **word** -> **Root Word**

**Example:**


1.   Actor, Acting -> **Act**
2.   Running, Runner -> **Run**

**Purpose of Stemming:** Mainly used for **Data Reduction** process in the **Data Processing** step. As the dataset is very huge, we reduce the words to their stem to feed it to the model and for ease of analysis by the model.



In [ ]:
porter_stem = PorterStemmer()
print("Porter Stem created and initialised with the Porter Stemmer Function")

Porter Stem created and initialised with the Porter Stemmer Function


In [ ]:
#Creating a function to reduce each word of the dataset to its stem in order to reduce the data)
def stemming(content):
  stemmed_content = re.sub('[^a-zA-Z]', ' ', content)
  stemmed_content = stemmed_content.lower() #Converting to lowercase
  stemmed_content = stemmed_content.split()
  stemmed_content = [porter_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content = ' '.join(stemmed_content)

  return stemmed_content

In [27]:
#Applying the function to the TEXT data in the dataset = Takes nearly 50mints to stem each word of 1.6million data

twitter_data['stemmed_content'] = twitter_data['text'].apply(stemming) #Creating a seperate column to store the stemmed content that is created by applying the stemming function on the text data


#Stemmed Content Column Added

In [28]:
#Printing First Five rows of data
twitter_data.head()

,target,id,date,flag,user,text,stemmed_content
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",switchfoot http twitpic com zl awww bummer sho...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,upset updat facebook text might cri result sch...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,kenichan dive mani time ball manag save rest g...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....",nationwideclass behav mad see


**We need JUST the Stemmed Content and Labels for model to understand and differentiate between positive and negative tweet**

In [31]:
print(twitter_data['stemmed_content']) #Printing the stemmed content indexed from 0 to 1599999

0          switchfoot http twitpic com zl awww bummer sho...
1          upset updat facebook text might cri result sch...
2          kenichan dive mani time ball manag save rest g...
3                            whole bodi feel itchi like fire
4                              nationwideclass behav mad see
                                 ...                        
1599995                           woke school best feel ever
1599996    thewdb com cool hear old walt interview http b...
1599997                         readi mojo makeov ask detail
1599998    happi th birthday boo alll time tupac amaru sh...
1599999    happi charitytuesday thenspcc sparkschar speak...
Name: stemmed_content, Length: 1600000, dtype: object


In [32]:
print(twitter_data['target']) # Printing the target or label (0 or 1)

0          0
1          0
2          0
3          0
4          0
          ..
1599995    1
1599996    1
1599997    1
1599998    1
1599999    1
Name: target, Length: 1600000, dtype: int64


In [34]:
#Separating the data and label
X=twitter_data['stemmed_content'].values
Y=twitter_data['target'].values

In [35]:
print(X) #will print first few tweets and last few tweets of the 1.6 million stemmed content

['switchfoot http twitpic com zl awww bummer shoulda got david carr third day'
 'upset updat facebook text might cri result school today also blah'
 'kenichan dive mani time ball manag save rest go bound' ...
 'readi mojo makeov ask detail'
 'happi th birthday boo alll time tupac amaru shakur'
 'happi charitytuesday thenspcc sparkschar speakinguph h']


In [36]:
print(Y) #first and last few targets will be printed

[0 0 0 ... 1 1 1]


**Splitting the data to training data and test data**

train_test_split function splits the data in training and test dataset
X_train,Y_train ->set of training data
X_test,Y_test->set of test data

test_size means how much data will go for training of the total data points we have here 20%

stratify=Y we want equal proportion of 0 and 1 class in both training and testing data

random_state=2 splitting will be replicated for same random state




In [37]:
X_train, X_test, Y_train, Y_test=train_test_split(X,Y,test_size=0.2, stratify=Y,random_state=2)

In [38]:
print(X.shape,X_train.shape,X_test.shape) #Printing size of data

(1600000,) (1280000,) (320000,)


In [39]:
print(X_train)

['watch saw iv drink lil wine' 'hatermagazin'
 'even though favourit drink think vodka coke wipe mind time think im gonna find new drink'
 ... 'eager monday afternoon'
 'hope everyon mother great day wait hear guy store tomorrow'
 'love wake folger bad voic deeper']


In [40]:
print(X_test)

['mmangen fine much time chat twitter hubbi back summer amp tend domin free time'
 'ah may show w ruth kim amp geoffrey sanhueza'
 'ishatara mayb bay area thang dammit' ...
 'destini nevertheless hooray member wonder safe trip' 'feel well'
 'supersandro thank']


Feature Extration(Vectorizer)

In [41]:
#Converting the textual data to numerical data which tells importance of each word
#Based on this importance the tweets will be classified

vectorizer=TfidfVectorizer()

X_train=vectorizer.fit_transform(X_train) #fit understand the type of data we are giving and transform assigns score to all individual words ie transform the word to number
X_test=vectorizer.transform(X_test) # fit not used here because in test data transform will happen on basis of training data

In [42]:
print(X_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9453092 stored elements and shape (1280000, 461488)>
  Coords	Values
  (0, 436713)	0.27259876264838384
  (0, 354543)	0.3588091611460021
  (0, 185193)	0.5277679060576009
  (0, 109306)	0.3753708587402299
  (0, 235045)	0.41996827700291095
  (0, 443066)	0.4484755317023172
  (1, 160636)	1.0
  (2, 109306)	0.4591176413728317
  (2, 124484)	0.1892155960801415
  (2, 407301)	0.18709338684973031
  (2, 129411)	0.29074192727957143
  (2, 406399)	0.32105459490875526
  (2, 433560)	0.3296595898028565
  (2, 77929)	0.31284080750346344
  (2, 443430)	0.3348599670252845
  (2, 266729)	0.24123230668976975
  (2, 409143)	0.15169282335109835
  (2, 178061)	0.1619010109445149
  (2, 150715)	0.18803850583207948
  (2, 132311)	0.2028971570399794
  (2, 288470)	0.16786949597862733
  (3, 406399)	0.29029991238662284
  (3, 158711)	0.4456939372299574
  (3, 151770)	0.278559647704793
  (3, 56476)	0.5200465453608686
  :	:
  (1279996, 318303)	0.21254698865277744
  (12

In [43]:
print(X_test)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2289192 stored elements and shape (320000, 461488)>
  Coords	Values
  (0, 15110)	0.1719352837797837
  (0, 31168)	0.1624772418052177
  (0, 67828)	0.26800375270827315
  (0, 106069)	0.36555450010904555
  (0, 132364)	0.255254889555786
  (0, 138164)	0.23688292264071406
  (0, 171378)	0.2805816206356074
  (0, 271016)	0.45356623916588285
  (0, 279082)	0.17825180109103442
  (0, 388348)	0.2198507607206174
  (0, 398906)	0.34910438732642673
  (0, 409143)	0.3143047059807971
  (0, 420984)	0.17915624523539805
  (1, 6463)	0.30733520460524466
  (1, 15110)	0.211037449588008
  (1, 145393)	0.575262969264869
  (1, 217562)	0.40288153995289894
  (1, 256777)	0.28751585696559306
  (1, 348135)	0.4739279595416274
  (1, 366203)	0.24595562404108307
  (2, 22532)	0.3532582957477176
  (2, 34401)	0.37916255084357414
  (2, 89448)	0.36340369428387626
  (2, 183312)	0.5892069252021465
  (2, 256834)	0.2564939661498776
  :	:
  (319994, 443794)	0.2782185641032538


**Training the Machine Learning Model**

Logisting Regression
Classifying data points into different classes here 2

In [44]:
model=LogisticRegression(max_iter=1000) #maximum iteration

In [45]:
#Finding relation between X and Y like what words lead to positive tweets and which ones lead to negative tweets
#Feeding Training twwets and their label to model to learn from it
model.fit(X_train,Y_train)

LogisticRegression(max_iter=1000)

**Model Evaluation**

Accuracy Score

In [46]:
#accuracy score on the training data
X_train_prediction=model.predict(X_train)  #Model will predict labels
training_data_accuracy=accuracy_score(Y_train,X_train_prediction) #Compare the prediction with true labels of training data

In [47]:
print('Accuracy score on the training data: ', training_data_accuracy)

Accuracy score on the training data:  0.79871953125


In [48]:
#accuracy score on the test data
X_test_prediction=model.predict(X_test)  #Model will predict labels
test_data_accuracy=accuracy_score(Y_test,X_test_prediction) #Compare the prediction with true labels of test data

In [49]:
print('Accuracy score on the test data: ', test_data_accuracy)

Accuracy score on the test data:  0.77668125


Model Accuray=77.6%

**Saving the Trained Model**

In [50]:
import pickle

In [51]:
filename='trained_model.sav'
pickle.dump(model,open(filename,'wb')) #Writing all parameters and results of the model in the file

Using the saved model for future predictions

In [52]:
#Loading the saved model
loaded_model=pickle.load(open('/content/trained_model.sav', 'rb')) #Reading the file

In [55]:
X_new=X_test[200]
print(Y_test[200])

prediction=loaded_model.predict(X_new)
print(prediction)

if(prediction[0]==0):
  print('Negative Tweet')
else:
  print('Positive Tweet')

1
[1]
Positive Tweet


In [56]:
X_new=X_test[3]
print(Y_test[3])  #to check if model is predicting correctly

prediction=loaded_model.predict(X_new)
print(prediction)

if(prediction[0]==0):
  print('Negative Tweet')
else:
  print('Positive Tweet')

0
[0]
Negative Tweet
